# 🧠 Resume Screening & Ranking System — ML Development Notebook

This notebook walks you through every step of building the ML pipeline.
Run cells one-by-one and read the explanations.

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Load Sample Data](#2-data)
3. [PDF Parsing Demo](#3-pdf)
4. [Text Preprocessing](#4-preprocessing)
5. [Skill Extraction](#5-skills)
6. [TF-IDF Vectorization](#6-tfidf)
7. [Cosine Similarity & Ranking](#7-similarity)
8. [Skill Gap Analysis](#8-gap)
9. [Full Pipeline Demo](#9-pipeline)
10. [Kaggle Dataset Usage](#10-kaggle)

## 1. Setup & Imports
Run this cell first to install and import everything.

In [ ]:
# Install required packages (run once)
# !pip install spacy nltk scikit-learn pdfplumber pandas numpy
# !python -m spacy download en_core_web_sm

import sys
import os

# Add project root to path so we can import our modules
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

# Standard data science imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# NLP
import spacy
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Text processing
import re
import string
from nltk.corpus import stopwords

print('✅ All imports successful!')
print(f'Python version: {sys.version}')

## 2. Load Sample Data

We'll use sample resume and job description texts to demonstrate the pipeline.

**Using Kaggle Dataset:**
1. Go to: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
2. Download `Resume.csv`
3. Place it in `sample_data/Resume.csv`
4. Uncomment the Kaggle loading code below

In [ ]:
# ============================================================
# SAMPLE RESUMES (for demonstration — no files needed)
# ============================================================

sample_resumes = [
    {
        'name': 'Alice Chen',
        'text': """
        Senior ML Engineer with 6 years experience. Expert in Python, TensorFlow,
        PyTorch, and scikit-learn. Built NLP systems using spaCy, BERT, and
        Hugging Face Transformers. Deployed models on AWS using Docker and
        Kubernetes. Strong in SQL, PostgreSQL, and Apache Spark. FastAPI REST APIs.
        MLflow for experiment tracking. Deep learning, neural networks, XGBoost.
        Computer vision with OpenCV. Data analysis with Pandas and NumPy.
        """
    },
    {
        'name': 'Bob Martinez',
        'text': """
        Java backend developer 5 years. Spring Boot, Hibernate, MySQL, Microservices.
        REST API design, Docker, Jenkins CI/CD. Some Python scripting experience.
        AWS deployment experience. Agile Scrum methodology. Git, Jira, Confluence.
        Recently learning machine learning and data science basics.
        """
    },
    {
        'name': 'Carol Patel',
        'text': """
        Data Scientist 4 years. Python specialist — pandas, numpy, matplotlib,
        scikit-learn, XGBoost. Statistical analysis and data visualization.
        SQL and PostgreSQL expert. Tableau dashboards. A/B testing and hypothesis
        testing. Some TensorFlow experience. Jupyter notebooks. Google Cloud Platform.
        Machine learning: regression, classification, clustering.
        """
    },
    {
        'name': 'David Kim',
        'text': """
        Full-stack developer. React, TypeScript, Node.js frontend expert.
        Python Django and Flask backend. MySQL database. Git, GitHub Actions.
        Docker containerization. Basic machine learning knowledge. 
        Good communication and teamwork skills.
        """
    },
    {
        'name': 'Eva Rodriguez',
        'text': """
        ML Research Engineer. PhD Computer Science. Deep expertise in PyTorch,
        TensorFlow, and Keras for deep learning research. NLP specialist:
        BERT, GPT, transformers, spaCy, NLTK. Computer vision OpenCV.
        Python, numpy, scipy, matplotlib. AWS and Azure cloud.
        Docker and Kubernetes. Published 3 ML papers. XGBoost, LightGBM.
        FastAPI model serving. Apache Spark for big data.
        """
    }
]

sample_jd = """
Senior Machine Learning Engineer
Requirements: Python, TensorFlow, PyTorch, scikit-learn, NLP, spaCy,
deep learning, neural networks, XGBoost, FastAPI, Flask, REST API,
PostgreSQL, SQL, Docker, Kubernetes, AWS, Git, GitHub, machine learning,
data science, MLflow, Apache Spark, computer vision, OpenCV.
Nice to have: React, TypeScript.
Strong communication and teamwork required.
"""

print(f'✅ Loaded {len(sample_resumes)} sample resumes')
print(f'✅ Job description: {len(sample_jd)} characters')

# ============================================================
# OPTIONAL: Load Kaggle dataset
# ============================================================
# Uncomment below if you have Resume.csv from Kaggle:

# kaggle_df = pd.read_csv('sample_data/Resume.csv')
# print(f'Kaggle dataset: {len(kaggle_df)} resumes')
# print(f'Categories: {kaggle_df["Category"].unique()}')
# print(kaggle_df.head())

## 3. PDF Parsing Demo

**pdfplumber** opens a PDF and extracts text from each page.
This is the first step in our pipeline.

In [ ]:
import pdfplumber

def extract_text_from_pdf(file_path):
    """
    Extract text from a PDF file using pdfplumber.
    
    pdfplumber reads each page of the PDF and extracts
    the text content as a string.
    """
    text = ''
    try:
        with pdfplumber.open(file_path) as pdf:
            print(f'PDF has {len(pdf.pages)} pages')
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + '\n'
                    print(f'  Page {i+1}: extracted {len(page_text)} chars')
    except Exception as e:
        print(f'Error: {e}')
    return text

# Demo — try with a real PDF file:
# text = extract_text_from_pdf('sample_data/sample_resume.pdf')
# print(text[:500])

# For this demo, we'll use our text samples
print('PDF parser ready!')
print('To test: uncomment the lines above and provide a PDF path')

## 4. Text Preprocessing

**Goal:** Convert messy resume text into clean, normalized tokens.

Steps:
1. Lowercase
2. Remove URLs, emails, special characters  
3. Remove stopwords
4. Lemmatize (optional but helpful)

In [ ]:
# Load spaCy
nlp = spacy.load('en_core_web_sm')
stop_words = set(stopwords.words('english'))

# Add resume-specific stopwords
stop_words.update(['resume', 'curriculum', 'vitae', 'experience', 'skill',
                   'year', 'work', 'working', 'responsible', 'proficient'])

def clean_text(text):
    """Step 1: Clean raw text."""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)                # Remove emails
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)            # Keep only letters
    text = re.sub(r'\s+', ' ', text)                     # Clean whitespace
    return text.strip()

def remove_stopwords(text):
    """Step 2: Remove stopwords."""
    words = text.split()
    return ' '.join([w for w in words if w not in stop_words and len(w) > 2])

def lemmatize(text):
    """Step 3: Reduce words to base form."""
    doc = nlp(text[:10000])  # Limit for speed
    return ' '.join([token.lemma_ for token in doc
                     if not token.is_punct and not token.is_space and len(token.lemma_) > 2])

def preprocess(text):
    """Full preprocessing pipeline."""
    text = clean_text(text)
    text = remove_stopwords(text)
    text = lemmatize(text)
    return text

# ---- DEMO ----
raw = sample_resumes[0]['text']
print('=== Preprocessing Demo ===')
print(f'\n📝 Raw text ({len(raw)} chars):')
print(raw[:200])

cleaned = clean_text(raw)
print(f'\n🧹 After clean_text ({len(cleaned)} chars):')
print(cleaned[:200])

no_stops = remove_stopwords(cleaned)
print(f'\n🚫 After stopword removal ({len(no_stops.split())} tokens):')
print(no_stops[:200])

lemmatized = lemmatize(no_stops)
print(f'\n✨ After lemmatization ({len(lemmatized.split())} tokens):')
print(lemmatized[:200])

## 5. Skill Extraction

We match text against a curated skills database using regex.
This is more reliable than NER for technical skills.

In [ ]:
# Skills database (subset for demonstration)
SKILLS_DATABASE = [
    # Programming
    'python', 'java', 'javascript', 'typescript', 'sql', 'r', 'scala',
    'bash', 'c++', 'c#', 'go', 'rust', 'html', 'css',
    # ML/AI
    'machine learning', 'deep learning', 'neural network', 'nlp',
    'natural language processing', 'computer vision',
    'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'xgboost',
    'lightgbm', 'huggingface', 'bert', 'gpt', 'spacy', 'nltk',
    'random forest', 'gradient boosting', 'svm', 'regression', 'classification',
    'clustering', 'opencv', 'transformers', 'mlflow',
    # Data
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'tableau', 'power bi',
    'apache spark', 'hadoop', 'etl', 'statistics',
    # Databases
    'postgresql', 'mysql', 'mongodb', 'redis', 'elasticsearch', 'sqlite',
    # Cloud & DevOps
    'aws', 'azure', 'google cloud', 'docker', 'kubernetes', 'git',
    'github', 'ci/cd', 'jenkins', 'terraform', 'fastapi', 'flask', 'django',
    # Frameworks
    'react', 'vue', 'angular', 'node.js', 'spring', 'rest api',
    # Soft skills
    'agile', 'scrum', 'leadership', 'communication', 'teamwork'
]

def extract_skills(text):
    """Extract skills from text by keyword matching."""
    text_lower = text.lower()
    found = set()
    for skill in SKILLS_DATABASE:
        # \b = word boundary — prevents matching 'r' inside 'for'
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, text_lower):
            found.add(skill)
    return found

# ---- Extract skills from all resumes ----
print('=== Skill Extraction Results ===\n')
for resume in sample_resumes:
    skills = extract_skills(resume['text'])
    print(f"📄 {resume['name']}: {len(skills)} skills")
    print(f"   {', '.join(sorted(skills)[:12])}{'...' if len(skills) > 12 else ''}")
    print()

# ---- Skills from JD ----
jd_skills = extract_skills(sample_jd)
print(f'\n📋 JD requires {len(jd_skills)} skills:')
print(', '.join(sorted(jd_skills)))

## 6. TF-IDF Vectorization

**TF-IDF** converts text into numbers. Each word gets a score based on:
- **TF (Term Frequency)**: How often does the word appear in THIS document?
- **IDF (Inverse Document Frequency)**: How rare is the word across ALL documents?

Words that are frequent in one document but rare overall score HIGH.
Words that appear everywhere (like 'the') score LOW.

In [ ]:
# Preprocess all texts
print('Preprocessing texts...')
processed_resumes = [(r['name'], preprocess(r['text'])) for r in sample_resumes]
processed_jd = preprocess(sample_jd)
print('✅ Preprocessing complete!\n')

# Build corpus (JD first, then resumes)
corpus = [processed_jd] + [text for _, text in processed_resumes]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # Consider word pairs like 'machine learning'
    sublinear_tf=True,     # Use log scaling for TF
    max_features=10000,    # Top 10k most important features
    max_df=0.85            # Ignore words in 85%+ of documents
)

# Fit and transform — learn vocabulary + convert to vectors
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f'TF-IDF Matrix shape: {tfidf_matrix.shape}')
print(f'  Rows: {tfidf_matrix.shape[0]} documents (1 JD + {len(sample_resumes)} resumes)')
print(f'  Cols: {tfidf_matrix.shape[1]} unique terms')

# Show top terms for the JD
feature_names = vectorizer.get_feature_names_out()
jd_vector = tfidf_matrix[0].toarray()[0]
top_indices = jd_vector.argsort()[-15:][::-1]

print('\n🔑 Top TF-IDF terms in Job Description:')
for i in top_indices:
    if jd_vector[i] > 0:
        print(f'  {feature_names[i]:<25} {jd_vector[i]:.4f}')

## 7. Cosine Similarity & Ranking

**Cosine Similarity** measures the angle between two vectors.
- Score = 1.0: Identical direction (perfect match)
- Score = 0.0: Perpendicular (no common terms)

We compute cosine similarity between the JD vector and each resume vector.

In [ ]:
# JD vector = first row (index 0)
jd_vector_2d = tfidf_matrix[0:1]  # Shape: (1, n_features)

# Resume vectors = rows 1 onwards
resume_vectors = tfidf_matrix[1:]  # Shape: (n_resumes, n_features)

# Compute cosine similarity
# Result shape: (1, n_resumes) → flatten to 1D array
similarities = cosine_similarity(jd_vector_2d, resume_vectors).flatten()

# Create results dataframe
results = pd.DataFrame({
    'Candidate': [name for name, _ in processed_resumes],
    'Score': similarities,
    'Score_%': (similarities * 100).round(1)
})

# Sort by score (highest first) and add rank
results = results.sort_values('Score', ascending=False).reset_index(drop=True)
results.index += 1  # Start rank from 1
results.index.name = 'Rank'

print('🏆 === CANDIDATE RANKING ===' )
print(results.to_string())

# ---- Visualize ----
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#7c6dfa', '#a78bfa', '#60a5fa', '#34d399', '#fbbf24']
bars = ax.barh(
    results['Candidate'][::-1],
    results['Score_%'][::-1],
    color=colors[:len(results)],
    height=0.6
)

# Add score labels
for bar, score in zip(bars, results['Score_%'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score}%', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Relevance Score (%)', fontsize=12)
ax.set_title('Resume Ranking vs Job Description\n(TF-IDF + Cosine Similarity)', 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, max(results['Score_%']) * 1.15)
ax.axvline(x=50, color='red', linestyle='--', alpha=0.4, label='50% threshold')
ax.legend()
ax.grid(axis='x', alpha=0.3)
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor('#111118')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
ax.spines[:].set_color('#333')

plt.tight_layout()
plt.savefig('sample_data/ranking_chart.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0f')
plt.show()
print('\n📊 Chart saved to sample_data/ranking_chart.png')

## 8. Skill Gap Analysis

For each candidate, we identify:
- ✅ **Matching skills**: skills in both resume and JD
- ❌ **Missing skills**: required by JD but not in resume
- 🌟 **Extra skills**: in resume but not required by JD

In [ ]:
def skill_gap_analysis(resume_skills, jd_skills):
    """Compute skill gap between a resume and job description."""
    matching = resume_skills & jd_skills
    missing  = jd_skills - resume_skills
    extra    = resume_skills - jd_skills
    match_pct = (len(matching) / len(jd_skills) * 100) if jd_skills else 0
    
    return {
        'matching': sorted(matching),
        'missing':  sorted(missing),
        'extra':    sorted(extra),
        'match_pct': round(match_pct, 1)
    }

# Show detailed analysis for the top 3 candidates
top3 = results.head(3)['Candidate'].tolist()

for name in top3:
    resume_text = next(r['text'] for r in sample_resumes if r['name'] == name)
    resume_skills = extract_skills(resume_text)
    gap = skill_gap_analysis(resume_skills, jd_skills)
    
    score = results.set_index('Candidate').loc[name, 'Score_%']
    
    print(f'\n{'='*55}')
    print(f'📄 {name} — Relevance: {score}% | Skill Match: {gap["match_pct"]}%')
    print(f'{'='*55}')
    print(f'✅ Matching ({len(gap["matching"])}): {', '.join(gap["matching"][:8]) or "None"}')
    print(f'❌ Missing  ({len(gap["missing"])}): {', '.join(gap["missing"][:8]) or "None"}')
    print(f'🌟 Extra    ({len(gap["extra"])}): {', '.join(gap["extra"][:5]) or "None"}')

## 9. Full Pipeline Demo

Putting it all together in one clean function.

In [ ]:
def run_full_pipeline(resumes, job_description):
    """
    Complete resume screening pipeline.
    
    Steps:
    1. Preprocess all texts
    2. Build TF-IDF vectors
    3. Compute cosine similarities
    4. Rank candidates
    5. Extract skills & skill gaps
    6. Return ranked results
    """
    print('🚀 Starting Resume Screening Pipeline...')
    
    # Step 1: Preprocess
    print('  Step 1: Preprocessing...')
    proc_jd = preprocess(job_description)
    proc_resumes = [(r['name'], preprocess(r['text'])) for r in resumes]
    
    # Step 2: TF-IDF
    print('  Step 2: Building TF-IDF vectors...')
    corpus = [proc_jd] + [text for _, text in proc_resumes]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=10000)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    print(f'     Matrix: {tfidf_matrix.shape[0]} docs × {tfidf_matrix.shape[1]} features')
    
    # Step 3 & 4: Similarity + Ranking
    print('  Step 3: Computing cosine similarities...')
    similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    # Step 5: Skill extraction
    print('  Step 4: Extracting skills...')
    jd_skills = extract_skills(job_description)
    
    # Build final results
    results_list = []
    for i, resume in enumerate(resumes):
        resume_skills = extract_skills(resume['text'])
        gap = skill_gap_analysis(resume_skills, jd_skills)
        results_list.append({
            'name': resume['name'],
            'score': round(float(similarities[i]), 4),
            'score_pct': round(float(similarities[i]) * 100, 1),
            'skills_found': len(resume_skills),
            'skill_gap': gap
        })
    
    # Sort by score
    results_list.sort(key=lambda x: x['score'], reverse=True)
    for i, r in enumerate(results_list):
        r['rank'] = i + 1
    
    print('  ✅ Pipeline complete!')
    return results_list


# Run the pipeline!
final_results = run_full_pipeline(sample_resumes, sample_jd)

print('\n' + '='*55)
print('🏆 FINAL RANKED RESULTS')
print('='*55)
for r in final_results:
    emoji = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣'][r['rank'] - 1]
    print(f"{emoji} #{r['rank']} {r['name']}")
    print(f"   Relevance: {r['score_pct']}% | Skills: {r['skills_found']} found")
    print(f"   Skill Match: {r['skill_gap']['match_pct']}%")
    print(f"   Missing: {', '.join(r['skill_gap']['missing'][:5]) or 'None'}")
    print()

## 10. Kaggle Dataset Usage

Using the [Kaggle Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset)

In [ ]:
# ============================================================
# KAGGLE DATASET USAGE
# Download from: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
# Place Resume.csv in sample_data/ folder
# Then uncomment and run this cell
# ============================================================

# import pandas as pd
# 
# # Load the dataset
# df = pd.read_csv('sample_data/Resume.csv')
# 
# print('Dataset shape:', df.shape)
# print('Columns:', df.columns.tolist())
# print('\nCategories:')
# print(df['Category'].value_counts())
# 
# # Filter for a specific category (e.g., 'DATA SCIENCE')
# category = 'DATA SCIENCE'
# filtered = df[df['Category'] == category].head(20)
# 
# # Convert to our resume format
# kaggle_resumes = [
#     {'name': f'Candidate_{i}', 'text': row['Resume_str']}
#     for i, row in filtered.iterrows()
# ]
# 
# # Run the pipeline on Kaggle data!
# kaggle_jd = """
# Data Scientist position. Requirements: Python, machine learning, deep learning,
# scikit-learn, TensorFlow, pandas, numpy, SQL, statistical analysis,
# data visualization, matplotlib, AWS.
# """
# 
# kaggle_results = run_full_pipeline(kaggle_resumes, kaggle_jd)
# 
# print('Top 5 Data Science Candidates:')
# for r in kaggle_results[:5]:
#     print(f"  #{r['rank']} {r['name']}: {r['score_pct']}%")

print('👆 Uncomment the code above after downloading the Kaggle dataset!')
print('   URL: https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset')